In [8]:
import pandas as pd

def read_csv_to_dataframe(file_path: str) -> pd.DataFrame:
    """
    Reads a CSV file and converts it into a pandas DataFrame.
    
    Parameters:
        file_path (str): The path to the CSV file.
        
    Returns:
        pd.DataFrame: A DataFrame containing the CSV data.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return pd.DataFrame()

In [9]:
Rosedata_df = read_csv_to_dataframe("Event_core_v4.csv")

In [10]:
print(Rosedata_df.columns)

Index(['event_date', 'event_id', 'parent_event_id', 'type', 'location_id',
       'decimal_latitud', 'decimal_longitude', 'depth_m', 'habitat',
       'conteo_ind', 'temperat_grad_c', 'total_organic_matter_percent',
       'proport_silt_clay_percent'],
      dtype='object')


In [13]:
print(Rosedata_df.head())

   event_date   event_id parent_event_id    type location_id  decimal_latitud  \
0  30/05/2014  GG_GG01R1              GG  sample        GG01        22.239950   
1  30/05/2014  GG_GG01R2              GG  sample        GG01        22.239950   
2  30/05/2014  GG_GG01R3              GG  sample        GG01        22.239950   
3  31/05/2014  GG_GG02R1              GG  sample        GG02        22.430300   
4  03/06/2014  GG_GG04R1              GG  sample        GG04        22.306733   

   decimal_longitude  depth_m     habitat  conteo_ind  temperat_grad_c  \
0         -84.732217       20  coral reef         133             28.1   
1         -84.732217       20  coral reef         180             28.1   
2         -84.732217       20  coral reef          51             28.1   
3         -84.538383       18  coral reef         139             28.1   
4         -84.672667       18  coral reef         336             28.0   

   total_organic_matter_percent  proport_silt_clay_percent  
0      

Quiero agrupar por habitat y parent_event_id, y calcular la media y desviación estándar de temperat_grad_c y de total_organic_matter_percent:

In [20]:
result = (
    Rosedata_df.groupby(["habitat", "parent_event_id"])
      .agg({
          "total_organic_matter_percent": ["mean", "std"],
          "temperat_grad_c": ["mean", "std"]
      })
      .reset_index()
)

In [21]:
result

habitat parent_event_id total_organic_matter_percent            \
                                                           mean       std   
0       coral reef              GB                     8.450000  0.492950   
1       coral reef              GG                     8.842857  0.716140   
2  seagrass meadow              GB                     9.500000  0.766812   
3  seagrass meadow              GG                     9.450000  3.019290   

  temperat_grad_c            
             mean       std  
0       27.850000  0.273861  
1       28.057143  0.053452  
2       27.000000  0.000000  
3       28.750000  0.634648

In [25]:
import pandas as pd
from tabulate import tabulate

In [27]:
# Definir las variables a formatear
variables = ["total_organic_matter_percent", "temperat_grad_c"]

# Asignar nombres de columnas adecuados
result.columns = ['habitat', 'parent_event_id'] + [f"{var}_{stat}" for var in variables for stat in ['mean', 'std']]

# Formatear los valores para mostrar "media ± desviación estándar" sin notación científica y con dos cifras decimales
for var in variables:
    mean_col = f"{var}_mean"
    std_col = f"{var}_std"
    result[f"{var}_formatted"] = result.apply(lambda row: f"{row[mean_col]:,.2f} ± {row[std_col]:,.2f}", axis=1)

In [28]:
# Seleccionar solo las columnas formateadas junto con los índices
formatted_result = result[['habitat', 'parent_event_id'] + [f"{var}_formatted" for var in variables]]

# Mostrar la tabla con los resultados formateados usando tabulate
table = tabulate(formatted_result, headers='keys', tablefmt='pretty', showindex=False)
print(table)

+-----------------+-----------------+----------------------------------------+---------------------------+
|     habitat     | parent_event_id | total_organic_matter_percent_formatted | temperat_grad_c_formatted |
+-----------------+-----------------+----------------------------------------+---------------------------+
|   coral reef    |       GB        |              8.45 ± 0.49               |       27.85 ± 0.27        |
|   coral reef    |       GG        |              8.84 ± 0.72               |       28.06 ± 0.05        |
| seagrass meadow |       GB        |              9.50 ± 0.77               |       27.00 ± 0.00        |
| seagrass meadow |       GG        |              9.45 ± 3.02               |       28.75 ± 0.63        |
+-----------------+-----------------+----------------------------------------+---------------------------+


In [29]:
# Guardar la tabla en un archivo CSV
formatted_result.to_csv("resultados_agrupados.csv", index=False)

In [30]:
# Guardar la tabla en un archivo xlsx
formatted_result.to_excel("resultados_agrupados_excel.xlsx", index=False)